# 18 — One model, three inference paths: Qantara

**Lesson type: CPU teaching analogue.** This notebook explains the idea in the Qantara paper. It does not implement Qantara or reproduce a paper result.

## Learning goals

- Explain how one JEPA checkpoint supports three inference paths.
- Compare the observations and compute each path needs at inference time.
- Run a small deterministic toy example and keep it separate from paper measurements.

## Before you start

Prerequisites: basic Python. No prior JEPA or flow-matching experience is required.

Required tools: Python 3.10–3.13; the notebook uses the standard library only. It needs no GPU, data, checkpoint, download, or robot hardware.

Expected output: a printed map of the inference edges and fixed-seed toy outputs for the three paths. These toy outputs are not paper measurements.


## Paper context

Qantara trains an approximately 21-million-parameter JEPA to predict actions and future latents together. At inference time, one checkpoint can be queried as a planner, a behaviour-cloning policy, or a video–inverse policy. The choice depends on whether deployment can provide a goal observation and how much inference time it can spend.

The paper's central claim is about these different inference paths from the same checkpoint. The Push-T results below are paper values. Nothing in the local toy example measures those values.

Paper: [Qantara, arXiv v2](https://arxiv.org/abs/2607.04978).


In [ ]:
import platform
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Compute:", "CPU teaching example")


## 1. What bridge-flow training learns

An encoder maps each image to a latent `z_t`. Given the observed prefix, the predictor models the next action `a_t` and next latent `z_(t+1)` as a joint block.

The two tokens use different paths between a source and a clean target:

- **Action token:** noise-to-data interpolation, `a_tilde = tau_a * a + (1 - tau_a) * noise`.
- **State token:** a Brownian bridge from the known previous latent `z_t` toward `z_(t+1)`. The source at `tau_z = 0` is `z_t`, which is also available during rollout.

The state head predicts a residual from `z_t`; the action head predicts a flow velocity. SIGReg regularizes the clean latent distribution to reduce collapse. Training draws noise times on the four edges of the `(tau_a, tau_z)` square and on a diagonal mode. The edge samples train the regions queried by the inference paths.

This description follows Sections 2.1–2.3 of the paper. The next cell maps those query regions; it is not a model implementation.


In [ ]:
# Conceptual boundary map for action-noise and state-noise axes.
noise_modes = [
    ("forward", "tau_a = 1", "tau_z: 0 -> 1", "planning rollout"),
    ("policy", "tau_a: 0 -> 1", "tau_z = 0", "BC action sample"),
    ("video", "tau_a = 0", "tau_z: 0 -> 1", "action-free next latent"),
    ("inverse", "tau_a: 0 -> 1", "tau_z = 1", "action for a latent transition"),
    ("joint", "tau_a = tau_z: 0 -> 1", "diagonal", "training co-regularization"),
]
print(f"{'mode':<10} {'action axis':<23} {'state axis':<20} use")
for mode, action_axis, state_axis, use in noise_modes:
    print(f"{mode:<10} {action_axis:<23} {state_axis:<20} {use}")


## 2. Three inference paths from one checkpoint

| Path | Inputs at inference | Query | Paper setting | Main trade-off |
| --- | --- | --- | --- | --- |
| Latent planning | Current context and a goal observation | Use the forward edge to predict candidate outcomes; use CEM to search action sequences | `K=4` latent denoising steps | Can use the goal directly, but spends compute on search and rollouts |
| Behaviour cloning (BC) | Current context; no goal observation | Sample an action on the policy edge | `K=2` flow steps | Low inference cost; action quality depends on the demonstrations |
| Video–inverse | Current context; no goal observation | Predict an action-marginal next latent on the video edge, then recover an action on the inverse edge | `K=2` flow steps | Low inference cost; the predicted transition follows the data action prior |

In this paper, **video** means a predicted latent, not decoded pixels. Video–inverse inference first predicts the next latent without action conditioning. It then uses inverse dynamics to recover an action that bridges the current and predicted latents.

Only the planner receives a goal observation. The two imitation paths are goal-blind. This is a difference in available information, not only in runtime.


## 3. A small deterministic teaching analogue

The next cells use a scalar latent `z` and action `a` with the known transition `z_next = z + a`. One toy object exposes four query functions: forward, policy, video, and inverse.

The planner uses a small CEM-like search against a supplied goal. The BC-like path samples from a fixed toy action distribution without a goal. The video–inverse path predicts the mean next latent under that toy action distribution, then recovers the action from the latent difference.

These functions are **not** a learned JEPA, bridge-flow training, Qantara code, or a benchmark. They are a small analogy for the different inputs and queries.


In [ ]:
import math
import random
from inspect import signature


class ToyWorldModel:
    # An idealized analogy with scalar latents and known transitions.
    action_mean = 0.55

    def forward_edge(self, latent, action):
        return latent + action

    def policy_edge(self, latent, rng):
        # This signature has no goal argument: the path is goal-blind.
        del latent
        return max(-1.0, min(1.0, rng.gauss(self.action_mean, 0.08)))

    def video_edge(self, latent):
        # Predict the mean next latent under the toy action distribution.
        return latent + self.action_mean

    def inverse_edge(self, latent, next_latent):
        return next_latent - latent


def toy_cem(model, start, goal, seed=2026, horizon=3, population=160,
            elite_count=20, iterations=4):
    # Small CEM-like search; this is not Qantara's planner.
    if not 0 < elite_count <= population:
        raise ValueError("elite_count must be between 1 and population")
    rng = random.Random(seed)
    means = [0.0] * horizon
    deviations = [0.8] * horizon

    for _ in range(iterations):
        proposals = []
        for _ in range(population):
            actions = [
                max(-1.0, min(1.0, rng.gauss(mean, deviation)))
                for mean, deviation in zip(means, deviations)
            ]
            predicted = start
            for action in actions:
                predicted = model.forward_edge(predicted, action)
            proposals.append(((predicted - goal) ** 2, actions))

        proposals.sort(key=lambda item: item[0])
        elites = proposals[:elite_count]
        means = [
            sum(actions[index] for _, actions in elites) / elite_count
            for index in range(horizon)
        ]
        deviations = [
            max(
                0.05,
                math.sqrt(
                    sum((actions[index] - means[index]) ** 2 for _, actions in elites)
                    / elite_count
                ),
            )
            for index in range(horizon)
        ]

    planned_terminal = start
    for action in means:
        planned_terminal = model.forward_edge(planned_terminal, action)
    return {
        "first_action": means[0],
        "planned_terminal": planned_terminal,
        "terminal_cost": (planned_terminal - goal) ** 2,
    }


In [ ]:
toy_model = ToyWorldModel()
start_latent = 0.0
requested_goal = 2.4

planner_result = toy_cem(toy_model, start_latent, requested_goal)
planner_repeat = toy_cem(toy_model, start_latent, requested_goal)
assert planner_result == planner_repeat, "The fixed-seed toy search must be deterministic."

bc_action = toy_model.policy_edge(start_latent, random.Random(19))
bc_repeat = toy_model.policy_edge(start_latent, random.Random(19))
assert bc_action == bc_repeat, "The fixed-seed toy action sample must be deterministic."

video_next_latent = toy_model.video_edge(start_latent)
video_inverse_action = toy_model.inverse_edge(start_latent, video_next_latent)
assert math.isclose(video_inverse_action, toy_model.action_mean)
assert "goal" not in signature(toy_model.policy_edge).parameters
assert "goal" not in signature(toy_model.video_edge).parameters
assert "goal" not in signature(toy_model.inverse_edge).parameters

print("TOY ANALOGY OUTPUTS — not paper measurements")
print(
    "CEM-like planner:",
    f"goal={requested_goal:.2f}, first_action={planner_result['first_action']:.3f},",
    f"predicted_terminal={planner_result['planned_terminal']:.3f},",
    f"toy_cost={planner_result['terminal_cost']:.6f}",
)
print(f"BC-like sample: no goal input, action={bc_action:.3f}")
print(
    "Video-inverse:",
    f"no goal input, predicted_next_latent={video_next_latent:.3f},",
    f"recovered_action={video_inverse_action:.3f}",
)
print("Determinism and goal-blind signature checks passed.")


The output illustrates the inputs each path receives:

- The toy planner uses the requested goal to rank action sequences.
- The toy BC path samples an action without receiving a goal.
- The toy video–inverse path predicts a next latent from its action prior, then computes the action between the two toy latents.

The toy values are only outputs of this small analogy. They are not Qantara success rates, learned latents, or evidence that any path controls a robot.


## 4. Paper result: Push-T Table 2

These are values reported by the paper, not local measurements. The study uses three training seeds, five evaluation seeds, 50 episodes per evaluation seed, and a 50-step episode budget. The reported standard deviation is across the three training-seed means.

| Inference path | Push-T success rate reported in the paper |
| --- | ---: |
| CEM planning | 90.1 ± 1.1% |
| Behaviour cloning | 82.1 ± 1.0% |
| Video–inverse | 83.2 ± 0.6% |

The paper evaluates all three paths from the same checkpoint family. It does not establish that Qantara is the strongest method on every task. Its contribution here is the option to select different inference paths from one trained model.


## 5. What this lesson does and does not show

This local lesson shows how goal access and inference queries differ. Its scalar toy has a known transition and no images, learned encoder, training data, bridge-flow objective, or trained predictor. It cannot validate the paper's success rates or show that the paths work on Push-T.

The optional released-checkpoint evaluation is a separate feature path. It remains gated on verified data provenance, checkpoint handling, storage, compatible software, and a working headless renderer. This notebook downloads no data or weights.

### Sources

- Rakhimov et al., [Qantara: Bridge-Flow Training for Multi-Paradigm JEPA Inference](https://arxiv.org/abs/2607.04978), version 2.
- Authors' [official code at the reviewed revision](https://github.com/corl-team/qantara/tree/12429541011e8576eb4fe2bc589d059b0d29dd2e).
- See the project [preflight report](../docs/reproductions/qantara-preflight.md) for the separate data, checkpoint, and renderer gates.

## Check your understanding

1. Which paths receive a goal observation?
2. What does the video edge predict, and why does the inverse edge run after it?
3. Why can BC quality depend on the demonstration actions?
4. Which parts of the Qantara paper are absent from the scalar toy?
